# Explorer AquaMonitor-JYU

Ce notebook sert à comprendre le petit dataset JYU avant de l'utiliser.

Une ligne du fichier Parquet correspond à une image. Un même organisme peut avoir
plusieurs images : on utilise donc la colonne `individual` pour compter les vrais
individus.

Le notebook vérifie les volumes, les classes, les splits et quelques propriétés
des images. Les images elles-mêmes ne sont chargées que si on le demande.

## 1. Préparer le notebook

On définit ici les chemins et les paramètres principaux.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download

# Configuration centralisée : modifier ces constantes plutôt que les cellules d'analyse.
SEED = 42
DATASET_ID = "mikkoim/aquamonitor-jyu"
METADATA_FILENAME = "aquamonitor-jyu.parquet.gzip"
SPLIT_COLUMN = "fold0"
LABEL_COLUMN = "taxon_group"
LOAD_IMAGES = False  # passer à True pour afficher quelques images

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

# Résolution indépendante du répertoire depuis lequel Jupyter a été lancé.
repo_root = Path.cwd().resolve()
while repo_root.parent != repo_root and not (repo_root / "main.py").exists():
    repo_root = repo_root.parent

experiment_dir = repo_root / "experiments" / "aquamonitor"
data_dir = experiment_dir / "data"
reports_dir = experiment_dir / "reports"
# Création idempotente des répertoires de travail et de livraison.
data_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

print("Racine du dépôt :", repo_root)
print("Données locales   :", data_dir)
print("Rapports générés  :", reports_dir)

## 2. Charger les métadonnées

On télécharge seulement le petit fichier Parquet. Les images ne sont pas téléchargées.

In [ ]:
# Le client réutilise le cache local si la même révision est déjà disponible.
metadata_path = hf_hub_download(
    repo_id=DATASET_ID,
    filename=METADATA_FILENAME,
    repo_type="dataset",
    local_dir=data_dir / "metadata",
)

print(metadata_path)

# Format entrant : Parquet compressé, une ligne par image. La colonne
# individual relie les multiples images d'un même organisme.
# Le Parquet conserve les types et évite le coût mémoire d'une conversion CSV.
df = pd.read_parquet(metadata_path)
print("Fichier :", metadata_path)
print("Dimensions (lignes, colonnes) :", df.shape)
df.head()

## 3. Vérifier les colonnes

Cette vérification évite de continuer avec un fichier incomplet ou différent de celui attendu.

In [ ]:
# Contrat de données minimal requis par toutes les analyses suivantes.
required_columns = {
    "img", "individual", "imaging_run", LABEL_COLUMN, SPLIT_COLUMN,
    "camera", "width", "height",
}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Colonnes manquantes : {sorted(missing_columns)}"

print("Colonnes disponibles :")
print(df.columns.tolist())
print("\nValeurs manquantes dans les colonnes essentielles :")
display(df[list(required_columns)].isna().sum().sort_values(ascending=False))

## 4. Résumer les splits

On compte séparément les images, les acquisitions, les individus et les classes.

In [ ]:
# Agrégation simultanée des unités image, individu, acquisition et classe.
split_summary = (
    df.groupby(SPLIT_COLUMN)
    .agg(
        n_images=("img", "count"),
        n_individuals=("individual", "nunique"),
        n_imaging_runs=("imaging_run", "nunique"),
        n_classes=("taxon_group", "nunique"),
    )
    .reindex(["train", "val", "test"])
)
display(split_summary)

# Garde-fous de dérive : à réviser explicitement lors d'un changement de version du dataset.
assert set(df[SPLIT_COLUMN].dropna().unique()) == {"train", "val", "test"}
assert split_summary.loc["train", "n_images"] == 40880
assert split_summary.loc["val", "n_images"] == 6394
assert split_summary.loc["test", "n_images"] == 11192

## 5. Étudier les classes

Le nombre d'images peut être trompeur. Le nombre d'individus est plus important pour mesurer la diversité biologique.

In [ ]:
train_df = df[df[SPLIT_COLUMN] == "train"].copy()

# Le nombre d'individus est suivi séparément du nombre d'images pour éviter
# qu'une longue séquence d'un même organisme ne masque le déséquilibre réel.
class_stats = (
    train_df.groupby(LABEL_COLUMN)
    .agg(
        n_images=("img", "count"),
        n_individuals=("individual", "nunique"),
        n_imaging_runs=("imaging_run", "nunique"),
    )
    .sort_values("n_individuals", ascending=False)
)
class_stats["images_per_individual"] = (
    class_stats["n_images"] / class_stats["n_individuals"]
)

display(class_stats)
class_stats.to_csv(reports_dir / "class_statistics.csv")

In [ ]:
# Tri croissant adapté à la lecture d'un diagramme horizontal.
ordered = class_stats.sort_values("n_individuals")
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(ordered.index, ordered["n_individuals"], color="#2878B5")
ax.set_title("AquaMonitor-JYU / individus d'entraînement par classe")
ax.set_xlabel("Nombre d'individus distincts")
ax.set_ylabel("Taxon group")
fig.tight_layout()
fig.savefig(reports_dir / "class_distribution_individuals.png", dpi=160)
plt.show()

## 6. Vérifier les fuites

Un même individu ne doit pas apparaître dans deux splits différents.

In [ ]:
def overlap_count(column: str, split_a: str, split_b: str) -> int:
    """Count identifiers shared by two dataset splits.

    Args:
        column: Identifier column to audit (image, individual or imaging run).
        split_a: First split label.
        split_b: Second split label.

    Returns:
        Number of distinct identifiers present in both splits.

    This control detects leakage that would overestimate model performance.
    """
    values_a = set(df.loc[df[SPLIT_COLUMN] == split_a, column])
    values_b = set(df.loc[df[SPLIT_COLUMN] == split_b, column])
    return len(values_a & values_b)

# Les trois paires couvrent exhaustivement train/validation/test.
overlap_rows = []
for split_a, split_b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap_rows.append({
        "comparison": f"{split_a} / {split_b}",
        "shared_individuals": overlap_count("individual", split_a, split_b),
        "shared_imaging_runs": overlap_count("imaging_run", split_a, split_b),
        "shared_images": overlap_count("img", split_a, split_b),
    })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
assert overlap_df[["shared_individuals", "shared_imaging_runs", "shared_images"]].to_numpy().sum() == 0

## 7. Regarder les propriétés des images

On vérifie notamment la caméra et les dimensions, car elles peuvent créer des biais.

In [ ]:
display(train_df[["width", "height", "area", "perimeter"]].describe())
display(train_df["camera"].value_counts(dropna=False).sort_index())

# Normalisation par ligne : chaque cellule représente une proportion au sein de la classe.
camera_by_class = pd.crosstab(
    train_df[LABEL_COLUMN],
    train_df["camera"],
    normalize="index",
)

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(camera_by_class, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title("Proportion des images de chaque caméra par classe")
ax.set_xlabel("Caméra")
ax.set_ylabel("Taxon group")
fig.tight_layout()
fig.savefig(reports_dir / "camera_distribution_by_class.png", dpi=160)
plt.show()

## 8. Enregistrer un résumé

Les résultats principaux sont sauvegardés dans un JSON et un CSV.

In [ ]:
# Résumé sérialisable destiné au suivi d'expérience et aux traitements automatisés.
summary = {
    "dataset_id": DATASET_ID,
    "metadata_filename": METADATA_FILENAME,
    "split_column": SPLIT_COLUMN,
    "label_column": LABEL_COLUMN,
    "n_rows": int(len(df)),
    "n_classes": int(df[LABEL_COLUMN].nunique()),
    "splits": {
        split: {key: int(value) for key, value in row.items()}
        for split, row in split_summary.to_dict(orient="index").items()
    },
    "split_overlap": overlap_rows,
}

summary_path = reports_dir / "dataset_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(summary_path)
summary

## 9. Afficher quelques images (optionnel)

Pour activer cette partie, mettre `LOAD_IMAGES = True` dans la première cellule,
puis relancer le notebook. Le mode streaming évite de télécharger tout le dataset.

In [ ]:
image_samples = []

# Branche optionnelle et coûteuse : elle ne participe à aucun KPI principal.
if LOAD_IMAGES:
    from datasets import load_dataset

    image_stream = load_dataset(
        DATASET_ID,
        split="train",
        streaming=True,
        cache_dir=str(data_dir / "hf_cache"),
    )
    image_samples = list(image_stream.take(12))
    print("Clés du premier exemple :", image_samples[0].keys())
else:
    print("Images non chargées. Mettre LOAD_IMAGES = True pour activer cette partie.")

In [ ]:
if image_samples:
    metadata_by_image = df.set_index("img")
    fig, axes = plt.subplots(3, 4, figsize=(13, 8))

    for ax, sample in zip(axes.flat, image_samples):
# Détection robuste de la colonne image sans dépendre de son nom dans le flux.
        image_key = next(
            key for key, value in sample.items()
            if hasattr(value, "size") and hasattr(value, "convert")
        )
        image = sample[image_key]
        sample_key = sample.get("__key__", "")
        filename = sample_key if sample_key.endswith(".jpg") else f"{sample_key}.jpg"
        label = metadata_by_image.loc[filename, LABEL_COLUMN] if filename in metadata_by_image.index else "label inconnu"

        ax.imshow(image)
        ax.set_title(str(label), fontsize=9)
        ax.axis("off")

    fig.suptitle("Premières images du flux d'entraînement")
    fig.tight_layout()
    plt.show()
else:
    print("Aucun exemple chargé.")

## 10. Ce qu'il faut retenir

- une classe avec beaucoup d'images n'a pas forcément beaucoup d'individus ;
- les splits doivent être séparés au niveau des individus ;
- la caméra ou la taille des images peuvent devenir des raccourcis pour le modèle ;
- les tableaux produits ici servent de base aux notebooks de rapprochement taxonomique.